# Project 02 — Apriori-TID Mining

# 1. Khởi tạo môi trường

In [ ]:
import os
import shutil
import shlex
import re
import subprocess
import tempfile
from pathlib import Path
def download_runtime_archive(url, destination):
    subprocess.run(["curl", "-fLsS", "--retry", "2", "--max-time", "240",
                    url, "-o", str(destination)], check=True)
from IPython.display import Image, SVG, Markdown, display

repository_url = "https://github.com/torusama/apriori-tid-mining.git"
repository_branch = "main"

repository = None
for directory in [Path.cwd(), *Path.cwd().parents]:
    if (directory / "scripts" / "visual_pipeline.jl").is_file():
        repository = directory
        break

if repository is None:
    repository = Path.cwd() / "apriori-tid-mining"
    uploaded_zip = Path("/content/apriori-tid-mining-reviewed.zip")
    if not uploaded_zip.is_file():
        uploaded_zip = Path("/content/apriori-tid-mining-ready.zip")
    if not uploaded_zip.is_file():
        uploaded_zip = Path("/content/apriori-tid-mining.zip")
    if uploaded_zip.is_file():
        subprocess.run(["unzip", "-o", str(uploaded_zip), "-d", str(Path.cwd())], check=True)
    elif not repository.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", repository_branch,
                        repository_url, str(repository)], check=True)

for required_file in ["demo.jl", "scripts/fetch_datasets.jl", "scripts/plot_results.jl",
                      "scripts/visual_pipeline.jl", "scripts/run_all_tests.jl"]:
    if not (repository / required_file).is_file():
        raise RuntimeError("The repository is missing the integrated files. Upload the ZIP or update the GitHub branch.")

julia = shutil.which("julia")
if julia is None:
    julia_version = "1.10.10"
    installation_directory = Path(tempfile.gettempdir()) / ("julia-" + julia_version)
    julia = str(installation_directory / "bin" / "julia")
    if not Path(julia).is_file():
        with tempfile.TemporaryDirectory() as download_directory:
            archive = Path(download_directory) / "julia.tar.gz"
            download_runtime_archive(f"https://julialang-s3.julialang.org/bin/linux/x64/1.10/julia-{julia_version}-linux-x86_64.tar.gz", archive)
            subprocess.run(["tar", "-xzf", str(archive), "-C", str(installation_directory.parent)], check=True)

# Dùng Java 25+ cho SPMF; trên Colab tự cài JRE nếu cần.
def java_is_compatible():
    if shutil.which("java") is None:
        return False
    version = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
    major = re.search(r'version "(\d+)', version.stderr + version.stdout)
    return major is not None and int(major.group(1)) >= 25

if not java_is_compatible():
    if not Path("/content").is_dir():
        raise RuntimeError("Install Java 25+ and add it to PATH before running this notebook locally.")
    java_directory = Path(tempfile.gettempdir()) / "apriori-openjdk25"
    java_directory.mkdir(parents=True, exist_ok=True)
    if not (java_directory / "bin/java").is_file():
        with tempfile.TemporaryDirectory() as download_directory:
            archive = Path(download_directory) / "java25.tar.gz"
            java_url = "https://download.java.net/java/GA/jdk25.0.2/b1e0dfa218384cb9959bdcb897162d4e/10/GPL/openjdk-25.0.2_linux-x64_bin.tar.gz"
            download_runtime_archive(java_url, archive)
            subprocess.run(["tar", "-xzf", str(archive), "-C", str(java_directory),
                            "--strip-components=1"], check=True)
    os.environ["JAVA_HOME"] = str(java_directory)
    os.environ["PATH"] = str(java_directory / "bin") + os.pathsep + os.environ["PATH"]

if not java_is_compatible():
    raise RuntimeError("SPMF needs Java 25 or newer.")
subprocess.run(["java", "-version"], check=True)

workspace = Path(tempfile.mkdtemp(prefix="apriori-demo-"))
raw_data = repository / "data" / "raw"

def run_julia(environment, script, *arguments, extra_environment=None, threads=None):
    command = [julia, "--startup-file=no"]
    if threads is not None:
        command.append(f"--threads={threads}")
    command.extend([f"--project={repository / environment}", str(repository / script)])
    for argument in arguments:
        command.append(str(argument))
    print(shlex.join(command), flush=True)
    subprocess.run(command, cwd=repository, check=True, env=extra_environment)

def instantiate(environment):
    subprocess.run([julia, "--startup-file=no", f"--project={repository / environment}",
                    "-e", "using Pkg; Pkg.instantiate()"], cwd=repository, check=True,
                   env=os.environ | {"JULIA_NUM_PRECOMPILE_TASKS": "2"})

print(subprocess.check_output([julia, "--version"], text=True).strip())
print("Repository:", repository)
instantiate(".")
instantiate("evaluation_environment")
instantiate("visual_environment")


## 2. Kiểm tra Basic và Optimized
Cùng một dữ liệu, cùng minsup 60%. Hai file kết quả phải trùng hoàn toàn.

In [ ]:
for implementation in ("basic", "optimized"):
    run_julia(".", "demo.jl", "--minsup", "0.6", repository / "data/toy/sample.txt",
              "--output", workspace / f"{implementation}.txt", "--implementation", implementation)

basic_result = (workspace / "basic.txt").read_text()
assert basic_result == (workspace / "optimized.txt").read_text()
print(basic_result)


## 3. Kiểm thử
Chạy đúng điểm vào `julia --project=. test/runtests.jl` theo đề. Java 25+ được chuẩn bị ở bước 1; toàn bộ test core, SPMF và ảnh phải hoàn thành. Nếu lỗi, cell báo lỗi thay vì ghi nhận pass hoặc tự bỏ qua.


In [ ]:
run_julia(".", "test/runtests.jl")


## 4. Benchmark — tải dữ liệu và chạy lại
Tải bốn dataset FIMI, chuẩn hóa định dạng SPMF, kiểm tra số giao dịch, số item và SHA-256 so với dữ liệu thực nghiệm đã lưu. Không cần upload dataset gốc lên GitHub.


In [ ]:
run_julia("evaluation_environment", "scripts/fetch_datasets.jl", "benchmark", "--root", raw_data)
benchmark_input_directory = raw_data / "benchmark"


### 4.1. Kiểm tra trực tiếp trên bốn dataset
Chạy Basic và Optimized, so từng itemset và support với SPMF ở một mức minsup cao của mỗi dataset. Đây là kiểm tra nhanh trên dữ liệu thật, không thay thế toàn bộ thực nghiệm 5 mức minsup.


In [ ]:
benchmark_quick_cases = [
    ("chess", "0.96"),
    ("mushroom", "0.75"),
    ("retail", "0.05"),
    ("T10I4D100K", "0.045"),
]
spmf_jar = repository / "tools/spmf.jar"
for dataset_name, minimum_support in benchmark_quick_cases:
    print(f"\n{dataset_name}: minsup = {minimum_support}", flush=True)
    run_julia(".", "scripts/compare_spmf.jl", spmf_jar,
              benchmark_input_directory / f"{dataset_name}.txt", minimum_support, threads=1)


### 4.2. Tái tạo benchmark đầy đủ
Mặc định `run_full_benchmark = True` để chạy năm minsup trên mỗi dataset, peak RSS, scalability và ảnh hưởng độ dài giao dịch. Kết quả được tạo mới trong workspace; bước này có thể mất nhiều thời gian. Đặt `False` để bỏ qua benchmark đầy đủ và chạy nhanh các ứng dụng.

In [ ]:
run_full_benchmark = True  #@param {type:"boolean"}
if run_full_benchmark:
    benchmark_directory = workspace / "benchmark_new"
    run_julia(".", "scripts/run_benchmark.jl", benchmark_input_directory,
              benchmark_directory, spmf_jar, repository / "configs/benchmark.toml", threads=1)
    run_julia("evaluation_environment", "scripts/plot_results.jl", "benchmark", benchmark_directory)
    print("New benchmark results:", benchmark_directory)
    print((benchmark_directory / "environment.txt").read_text())
    for filename in ("runtime_ms.png", "itemsets.png", "peak_ram.png", "scaling_and_width.png"):
        display(Image(filename=str(benchmark_directory / filename)))
else:
    print("Skipped full benchmark; set run_full_benchmark = True to reproduce it.")


## 5. Ứng dụng 1 — Market Basket Analysis
Tải dữ liệu Groceries gốc, chuyển giỏ hàng thành transaction, khai thác bằng Apriori-TID và tạo luật kết hợp. Minsup 1%, minconf 40%.

In [ ]:
groceries_output = workspace / "groceries"
run_julia("evaluation_environment", "scripts/fetch_datasets.jl", "groceries", "--root", raw_data)
run_julia(".", "scripts/run_market_basket.jl", raw_data / "groceries/Groceries.data",
          raw_data / "groceries/Groceries.labels", groceries_output, "0.01", "0.4", "optimized")
run_julia("evaluation_environment", "scripts/plot_results.jl", "groceries", groceries_output)
print((groceries_output / "top10_rules.csv").read_text())
for filename in ("basket_lengths.png", "top10_rules.png"):
    display(Image(filename=str(groceries_output / filename)))

## 6. Ứng dụng 2 — Visual Pattern Mining
Oxford Flowers-17 → thu nhỏ ảnh → trích xuất ORB → K-Means tạo visual vocabulary → chuyển mỗi ảnh thành một transaction → Apriori-TID optimized → kiểm tra support và hiển thị ảnh mẫu.

Ảnh không có descriptor vẫn đóng góp vào mẫu số support. Chạy toàn bộ 1.360 ảnh (`image_limit = 0`). Có thể đặt giới hạn nhỏ để kiểm tra nhanh; dữ liệu giới hạn lấy theo thứ tự tên file, không đại diện đủ 17 lớp.

In [ ]:
image_limit = 0
visual_words = 8
maximum_descriptors = 100
minimum_support_values = "0.7,0.8,0.9"
visual_output = workspace / "flowers_k8"

run_julia("visual_environment", "scripts/visual_pipeline.jl", "--output", visual_output,
          "--limit", image_limit, "--words", visual_words, "--maximum-descriptors", maximum_descriptors,
          "--seed", 42, "--minsup", minimum_support_values)
print((visual_output / "dataset_summary.toml").read_text())
print((visual_output / "runs.csv").read_text())
display(Markdown((visual_output / "analysis.md").read_text()))
for figure_path in sorted((visual_output / "figures").glob("*.svg")):
    display(SVG(filename=str(figure_path)))

## 7. Thử cấu hình ảnh khác
Tăng số cụm lên 16, giảm số ORB mỗi ảnh còn 40 để xem các giao dịch có bớt bão hòa hay không. Giữ nguyên tập ảnh, seed và các minsup để đối chiếu. Hai tham số thay đổi đồng thời nên đây là so sánh cấu hình, không phải thí nghiệm tách riêng ảnh hưởng từng tham số.

In [ ]:
alternative_output = workspace / "flowers_k16"
run_julia("visual_environment", "scripts/visual_pipeline.jl", "--output", alternative_output,
          "--limit", image_limit, "--words", 16, "--maximum-descriptors", 40,
          "--seed", 42, "--minsup", minimum_support_values)
print((alternative_output / "dataset_summary.toml").read_text())
print((alternative_output / "runs.csv").read_text())
display(Markdown((alternative_output / "analysis.md").read_text()))
for figure_path in sorted((alternative_output / "figures").glob("*.svg")):
    display(SVG(filename=str(figure_path)))

## 8. Tổng kết
Groceries dùng luật kết hợp để gợi ý giả thuyết mua kèm, không chứng minh quan hệ nhân quả. Visual patterns có thể tạo đặc trưng nhị phân cho phân loại hoặc truy vấn ảnh; cần đo hiệu quả trên tập kiểm tra riêng. Kết quả mới nằm trong thư mục workspace được in ở dưới. Không sử dụng benchmark cũ làm bằng chứng kiểm thử của code mới.

In [ ]:
print("Output directory:", workspace)